Component ablation A3

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

acoustic_dataset_1 = pd.read_csv('/content/drive/MyDrive/Acoustic-subject-level-dataset.csv')
acoustic_dataset = acoustic_dataset_1.drop(columns=["Participant_ID"])
print(acoustic_dataset.shape)
print(acoustic_dataset.head())

(193, 70)
   MDVP:Fo(Hz)  MDVP:Fhi(Hz)  MDVP:Flo(Hz)  MDVP:Jitter(%)  MDVP:Jitter(Abs)  \
0   118.719333    141.312833    106.029833        0.010085          0.000087   
1   113.014000    154.201333    105.652667        0.008043          0.000072   
2   223.657833    239.034500    161.921500        0.002585          0.000010   
3   141.336667    165.070500     66.233833        0.009008          0.000065   
4   112.839667    208.860000    107.392833        0.004850          0.000042   

   MDVP:RAP  MDVP:PPQ  Jitter:DDP  MDVP:Shimmer  MDVP:Shimmer(dB)  ...  \
0  0.004998  0.007312    0.014992      0.053932          0.515167  ...   
1  0.003508  0.005035    0.010527      0.029677          0.288667  ...   
2  0.001463  0.001425    0.004393      0.017093          0.151333  ...   
3  0.005090  0.004368    0.015263      0.042640          0.407333  ...   
4  0.002468  0.002817    0.007405      0.025015          0.224500  ...   

   Delta4  Delta5  Delta6  Delta7  Delta8  Delta9  Delta10  Delt

In [3]:
import pandas as pd

meg_dataset = pd.read_csv('/content/drive/MyDrive/final_meg_rest_21_par-lev_final.csv')

print("Loaded data shape:", meg_dataset.shape)
print(meg_dataset.head())

Loaded data shape: (127, 106)
   burst_rate_L_mean  burst_rate_L_std  burst_rate_R_mean  burst_rate_R_std  \
0           4.412365          0.242544           4.486517          0.188241   
1           3.828010          0.316999           3.783657          0.297773   
2           4.645359          0.130894           4.713304          0.111269   
3           4.142208          0.243497           4.183916          0.217601   
4           3.923527          0.336133           4.035406          0.306960   

   burst_rate_asym  mean_burst_duration_L_mean  mean_burst_duration_L_std  \
0        -0.008333                    0.056832                   0.003237   
1         0.005827                    0.065725                   0.005382   
2        -0.007260                    0.053844                   0.001551   
3        -0.005009                    0.060555                   0.003554   
4        -0.014057                    0.064178                   0.005454   

   mean_burst_duration_R_mean  m

Experiment	Residual block	DANN / GRL	SCL	Purpose
A3. Residual encoder + DANN	✓	✓	✗	Effect of domain-adversarial learning

In [4]:
# ============================================================
# MIRA-NET COMPONENT ABLATION A3: RESIDUAL ENCODER + DANN
# Residual block ON | DANN/GRL ON | SCL OFF
#
# Pipeline retained:
#   1. Stratified 5-fold CV separately for acoustic and MEG
#   2. Train-fold-only clipping and median imputation
#   3. RF-SHAP Top-7 feature selection per modality per fold
#   4. StandardScaler fitted on training folds only
#   5. Shared residual encoder with 32-D normalized embeddings
#   6. DANN with Gradient Reversal Layer
#   7. Final MLP classifier
#   8. OOF predictions, metrics, and latent embedding exports
#
# A3 removes ONLY supervised contrastive learning (SCL).
#
# Assumptions:
#   acoustic_dataset: full-feature acoustic DataFrame
#   meg_dataset:      full-feature MEG DataFrame
#   Last column:      binary label (0=HC, 1=PD)
# ============================================================

import random
import warnings

import numpy as np
import pandas as pd
import shap

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore", category=UserWarning)


# ============================================================
# DEVICE AND REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", device)


# ============================================================
# MODEL SETTINGS
# ============================================================

LATENT_DIM = 32
TOP_K = 7
N_SPLITS = 5

# Random Forest for RF-SHAP
RF_N_ESTIMATORS = 300

# Numerical safety cap
NUMERICAL_CAP = 1e12

# Encoder and optimizer
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
ENCODER_EPOCHS = 120

# DANN task/domain loss weights
TASK_WEIGHT = 3.0
DOMAIN_WEIGHT = 0.25

# Gradient reversal schedule
GRL_MAX_LAMBDA = 0.25
GRL_WARMUP_EPOCHS = 50

# Final MLP
MLP_EPOCHS = 80
MLP_LEARNING_RATE = 1e-3


# ============================================================
# LOAD FULL-FEATURE DATASETS
# ============================================================

df_acoustic_full = acoustic_dataset.copy()
df_meg_full = meg_dataset.copy()


def extract_features_and_labels(df, dataset_name):

    if df.shape[1] < 2:
        raise ValueError(
            f"{dataset_name} must contain features and a label."
        )

    y = pd.to_numeric(
        df.iloc[:, -1],
        errors="raise"
    ).to_numpy(dtype=np.int64)

    X_df = df.iloc[:, :-1].apply(
        pd.to_numeric,
        errors="coerce"
    )

    X = X_df.to_numpy(dtype=np.float64)

    # Convert NaN and infinite values into missing values.
    X[~np.isfinite(X)] = np.nan

    if not np.isin(y, [0, 1]).all():
        raise ValueError(
            f"{dataset_name}: labels must be encoded as 0 and 1."
        )

    if X.shape[1] == 0:
        raise ValueError(
            f"{dataset_name}: no candidate features found."
        )

    class_counts = np.bincount(y, minlength=2)

    if len(y) < N_SPLITS or class_counts.min() < N_SPLITS:
        raise ValueError(
            f"{dataset_name}: each class must contain at least "
            f"{N_SPLITS} participants for stratified CV."
        )

    return X, y, list(X_df.columns)


(
    X_acoustic_raw,
    y_acoustic,
    acoustic_feature_names
) = extract_features_and_labels(
    df_acoustic_full,
    "Acoustic"
)

(
    X_meg_raw,
    y_meg,
    meg_feature_names
) = extract_features_and_labels(
    df_meg_full,
    "MEG"
)

print("\nDataset shapes")
print("Acoustic:", X_acoustic_raw.shape)
print("MEG     :", X_meg_raw.shape)

print(
    "Acoustic labels:",
    dict(zip(*np.unique(y_acoustic, return_counts=True)))
)

print(
    "MEG labels:",
    dict(zip(*np.unique(y_meg, return_counts=True)))
)


# ============================================================
# LEAKAGE-FREE NUMERICAL PREPROCESSING
# ============================================================

def preprocess_fold_train_test(X_train, X_test):

    X_train = np.asarray(
        X_train,
        dtype=np.float64
    ).copy()

    X_test = np.asarray(
        X_test,
        dtype=np.float64
    ).copy()

    if (
        X_train.ndim != 2
        or X_test.ndim != 2
        or X_train.shape[1] != X_test.shape[1]
    ):
        raise ValueError(
            "Train/test features must be 2-D with matching "
            "feature counts."
        )

    # Replace non-finite values.
    X_train[~np.isfinite(X_train)] = np.nan
    X_test[~np.isfinite(X_test)] = np.nan

    # Fixed numerical safety cap.
    X_train = np.clip(
        X_train,
        -NUMERICAL_CAP,
        NUMERICAL_CAP
    )

    X_test = np.clip(
        X_test,
        -NUMERICAL_CAP,
        NUMERICAL_CAP
    )

    n_features = X_train.shape[1]

    lower_bounds = np.zeros(
        n_features,
        dtype=np.float64
    )

    upper_bounds = np.zeros(
        n_features,
        dtype=np.float64
    )

    medians = np.zeros(
        n_features,
        dtype=np.float64
    )

    # Learn clipping bounds from training data only.
    for j in range(n_features):

        values = X_train[:, j]
        values = values[np.isfinite(values)]

        if values.size == 0:
            lower_bounds[j] = 0.0
            upper_bounds[j] = 0.0
        else:
            lower_bounds[j] = np.quantile(
                values,
                0.01
            )

            upper_bounds[j] = np.quantile(
                values,
                0.99
            )

    # Apply training-derived bounds to both sets.
    X_train = np.maximum(
        np.minimum(X_train, upper_bounds),
        lower_bounds
    )

    X_test = np.maximum(
        np.minimum(X_test, upper_bounds),
        lower_bounds
    )

    # Learn medians from the training data only.
    for j in range(n_features):

        values = X_train[:, j]
        values = values[np.isfinite(values)]

        medians[j] = (
            np.median(values)
            if values.size
            else 0.0
        )

    def impute_with_training_medians(X):

        X = np.asarray(
            X,
            dtype=np.float64
        ).copy()

        X[~np.isfinite(X)] = np.nan

        for j in range(X.shape[1]):

            missing = ~np.isfinite(X[:, j])

            X[missing, j] = medians[j]

        return X

    X_train_clean = impute_with_training_medians(
        X_train
    )

    X_test_clean = impute_with_training_medians(
        X_test
    )

    if (
        not np.isfinite(X_train_clean).all()
        or not np.isfinite(X_test_clean).all()
    ):
        raise ValueError(
            "Non-finite values remain after preprocessing."
        )

    return (
        X_train_clean,
        X_test_clean,
        medians,
        lower_bounds,
        upper_bounds
    )


# ============================================================
# FOLD-WISE RF-SHAP TOP-7 FEATURE SELECTION
# ============================================================

def select_top_features_rf_shap(
    X_train,
    y_train,
    feature_names,
    top_k=7,
    random_state=42,
    dataset_name="Dataset"
):

    X_train = np.asarray(
        X_train,
        dtype=np.float64
    )

    y_train = np.asarray(
        y_train,
        dtype=np.int64
    )

    if not np.isfinite(X_train).all():
        raise ValueError(
            f"{dataset_name}: non-finite values before RF fitting."
        )

    k = min(top_k, X_train.shape[1])

    if k < 1:
        raise ValueError(
            f"{dataset_name}: no features available."
        )

    rf = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        random_state=random_state,
        class_weight="balanced",
        n_jobs=-1
    )

    # Fit on the outer-training fold only.
    rf.fit(
        X_train,
        y_train
    )

    # Explain training-fold observations only.
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_train)

    # Support different SHAP output formats.
    if isinstance(shap_values, list):

        class_idx = (
            1 if len(shap_values) > 1 else 0
        )

        sv = np.asarray(
            shap_values[class_idx]
        )

    else:

        sv = np.asarray(shap_values)

        if sv.ndim == 3:

            # Format: samples x features x classes.
            if (
                sv.shape[0] == len(X_train)
                and sv.shape[1] == X_train.shape[1]
            ):

                class_idx = min(
                    1,
                    sv.shape[2] - 1
                )

                sv = sv[:, :, class_idx]

            # Format: classes x samples x features.
            elif (
                sv.shape[1] == len(X_train)
                and sv.shape[2] == X_train.shape[1]
            ):

                class_idx = min(
                    1,
                    sv.shape[0] - 1
                )

                sv = sv[class_idx, :, :]

            else:
                raise ValueError(
                    f"{dataset_name}: unexpected SHAP shape "
                    f"{sv.shape}."
                )

    if sv.shape != X_train.shape:
        raise ValueError(
            f"{dataset_name}: SHAP shape {sv.shape} does not "
            f"match training data shape {X_train.shape}."
        )

    if not np.isfinite(sv).all():
        raise ValueError(
            f"{dataset_name}: SHAP returned non-finite values."
        )

    # Mean absolute SHAP importance.
    importance = np.mean(
        np.abs(sv),
        axis=0
    )

    selected_indices = np.argsort(
        importance
    )[::-1][:k]

    selected_names = [
        feature_names[i]
        for i in selected_indices
    ]

    print(f"\n{dataset_name} RF-SHAP Top-{k} features:")

    for rank, idx in enumerate(
        selected_indices,
        start=1
    ):

        print(
            f"{rank:02d}. {feature_names[idx]} "
            f"(mean |SHAP|={importance[idx]:.6f})"
        )

    return selected_indices, selected_names


# ============================================================
# GRADIENT REVERSAL LAYER
# ============================================================

class GradReverse(torch.autograd.Function):

    @staticmethod
    def forward(ctx, x, lambd):

        ctx.lambd = float(lambd)

        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):

        return -ctx.lambd * grad_output, None


def grad_reverse(x, lambd):

    return GradReverse.apply(
        x,
        lambd
    )


# ============================================================
# RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(self, dim):

        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, x):

        return F.relu(
            self.block(x) + x
        )


# ============================================================
# A3 MODEL: RESIDUAL ENCODER + DANN
# SCL IS REMOVED
# ============================================================

class ResidualDANN(nn.Module):

    def __init__(self, input_dim, latent_dim=32):

        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),

            ResidualBlock(128),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Dropout(0.1),

            nn.Linear(64, latent_dim)
        )

        self.task_head = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(32, 2)
        )

        self.domain_head = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 2)
        )

    def forward(self, x, lambd=1.0):

        z = self.encoder(x)

        z_norm = F.normalize(
            z,
            p=2,
            dim=1
        )

        task_logits = self.task_head(z_norm)

        domain_logits = self.domain_head(
            grad_reverse(z_norm, lambd)
        )

        return task_logits, domain_logits, z_norm


# ============================================================
# GRL LAMBDA SCHEDULE
# ============================================================

def get_grl_lambda(epoch):

    progress = min(
        epoch / float(GRL_WARMUP_EPOCHS),
        1.0
    )

    lambd = GRL_MAX_LAMBDA * (
        2.0 / (1.0 + np.exp(-10.0 * progress)) - 1.0
    )

    return float(lambd)


# ============================================================
# TRAIN RESIDUAL ENCODER + DANN
# NO SCL LOSS IS USED
# ============================================================

def train_dann(
    X_acoustic_train,
    y_acoustic_train,
    X_meg_train,
    y_meg_train,
    fold_number
):

    if (
        X_acoustic_train.shape[1]
        != X_meg_train.shape[1]
    ):
        raise ValueError(
            "Acoustic and MEG must have equal selected "
            "feature counts for the shared encoder."
        )

    X_train_all = np.vstack([
        X_acoustic_train,
        X_meg_train
    ]).astype(np.float32)

    y_train_all = np.hstack([
        y_acoustic_train,
        y_meg_train
    ]).astype(np.int64)

    # Domain labels:
    # Acoustic = 0, MEG = 1.
    domain_train_all = np.hstack([
        np.zeros(
            len(y_acoustic_train),
            dtype=np.int64
        ),
        np.ones(
            len(y_meg_train),
            dtype=np.int64
        )
    ]).astype(np.int64)

    if not np.isfinite(X_train_all).all():
        raise ValueError(
            f"Non-finite DANN input in fold {fold_number}."
        )

    dataset = TensorDataset(
        torch.tensor(
            X_train_all,
            dtype=torch.float32
        ),
        torch.tensor(
            y_train_all,
            dtype=torch.long
        ),
        torch.tensor(
            domain_train_all,
            dtype=torch.long
        )
    )

    loader_generator = torch.Generator()
    loader_generator.manual_seed(
        SEED + fold_number
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        generator=loader_generator
    )

    model = ResidualDANN(
        input_dim=X_train_all.shape[1],
        latent_dim=LATENT_DIM
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    task_loss_fn = nn.CrossEntropyLoss()
    domain_loss_fn = nn.CrossEntropyLoss()

    print(
        f"\nTraining A3 Residual Encoder + DANN "
        f"(SCL OFF) | Fold {fold_number}"
    )

    for epoch in range(ENCODER_EPOCHS):

        model.train()

        total_loss = 0.0
        total_task_loss = 0.0
        total_domain_loss = 0.0

        total_correct = 0
        total_samples = 0

        lambd = get_grl_lambda(epoch)

        for xb, y_task, y_domain in loader:

            xb = xb.to(device)
            y_task = y_task.to(device)
            y_domain = y_domain.to(device)

            optimizer.zero_grad()

            task_logits, domain_logits, _ = model(
                xb,
                lambd=lambd
            )

            loss_task = task_loss_fn(
                task_logits,
                y_task
            )

            loss_domain = domain_loss_fn(
                domain_logits,
                y_domain
            )

            # A3 objective: task loss + domain loss.
            # SCL loss is deliberately absent.
            loss = (
                TASK_WEIGHT * loss_task
                +
                DOMAIN_WEIGHT * loss_domain
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite DANN loss in fold {fold_number}, "
                    f"epoch {epoch + 1}."
                )

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_task_loss += loss_task.item()
            total_domain_loss += loss_domain.item()

            predictions = task_logits.argmax(dim=1)

            total_correct += (
                predictions == y_task
            ).sum().item()

            total_samples += len(y_task)

        # Preserve the original logging frequency.
        if epoch == 0 or (epoch + 1) % 10 == 0:

            denominator = max(
                len(loader),
                1
            )

            train_accuracy = (
                total_correct / max(total_samples, 1)
            )

            print(
                f"Epoch {epoch + 1:03d}/{ENCODER_EPOCHS} "
                f"| Loss={total_loss / denominator:.4f} "
                f"| Task={total_task_loss / denominator:.4f} "
                f"| Domain={total_domain_loss / denominator:.4f} "
                f"| GRL={lambd:.3f} "
                f"| TrainAcc={train_accuracy:.4f}"
            )

    return model


# ============================================================
# EXTRACT 32-D LATENT FEATURES
# ============================================================

def extract_latent(model, X):

    X = np.asarray(
        X,
        dtype=np.float32
    )

    if not np.isfinite(X).all():
        raise ValueError(
            "Non-finite values passed to latent extraction."
        )

    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32,
        device=device
    )

    with torch.no_grad():

        _, _, z = model(
            X_tensor,
            lambd=0.0
        )

    z = z.cpu().numpy().astype(np.float32)

    if not np.isfinite(z).all():
        raise RuntimeError(
            "Non-finite latent embeddings generated."
        )

    return z


# ============================================================
# FINAL MLP CLASSIFIER
# ============================================================

class FinalMLP(nn.Module):

    def __init__(self, input_dim=32):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 2)
        )

    def forward(self, x):

        return self.net(x)


def train_and_evaluate_mlp(
    X_train,
    y_train,
    X_test,
    y_test,
    fold_number
):

    X_train = np.asarray(
        X_train,
        dtype=np.float32
    )

    X_test = np.asarray(
        X_test,
        dtype=np.float32
    )

    y_train = np.asarray(
        y_train,
        dtype=np.int64
    )

    y_test = np.asarray(
        y_test,
        dtype=np.int64
    )

    if (
        not np.isfinite(X_train).all()
        or not np.isfinite(X_test).all()
    ):
        raise ValueError(
            "Non-finite latent values passed to the MLP."
        )

    X_train_tensor = torch.tensor(
        X_train,
        dtype=torch.float32,
        device=device
    )

    X_test_tensor = torch.tensor(
        X_test,
        dtype=torch.float32,
        device=device
    )

    y_train_tensor = torch.tensor(
        y_train,
        dtype=torch.long,
        device=device
    )

    clf = FinalMLP(
        input_dim=LATENT_DIM
    ).to(device)

    optimizer_clf = optim.Adam(
        clf.parameters(),
        lr=MLP_LEARNING_RATE
    )

    criterion = nn.CrossEntropyLoss()

    # Preserve original full-batch MLP training.
    clf.train()

    for epoch in range(MLP_EPOCHS):

        optimizer_clf.zero_grad()

        outputs = clf(X_train_tensor)

        loss = criterion(
            outputs,
            y_train_tensor
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite MLP loss in fold {fold_number}."
            )

        loss.backward()
        optimizer_clf.step()

    # Evaluate held-out participants only.
    clf.eval()

    with torch.no_grad():

        outputs = clf(X_test_tensor)

        probabilities = F.softmax(
            outputs,
            dim=1
        )[:, 1].cpu().numpy()

        predictions = outputs.argmax(
            dim=1
        ).cpu().numpy()

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    metrics = {
        "Accuracy": accuracy_score(
            y_test,
            predictions
        ),

        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "Specificity": (
            tn / (tn + fp)
            if (tn + fp)
            else np.nan
        ),

        "F1": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),

        "AUC": (
            roc_auc_score(y_test, probabilities)
            if len(np.unique(y_test)) == 2
            else np.nan
        ),

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,

        "predictions": predictions,
        "probabilities": probabilities
    }

    print(f"\nA3 Fold {fold_number} held-out metrics:")

    for metric in [
        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1",
        "AUC"
    ]:

        print(
            f"{metric:12s}: {metrics[metric]:.4f}"
        )

    print(
        "Confusion matrix [[TN, FP], [FN, TP]]: "
        f"[[{tn}, {fp}], [{fn}, {tp}]]"
    )

    return metrics


# ============================================================
# OUTER STRATIFIED CROSS-VALIDATION SPLITS
# ============================================================

skf_acoustic = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

skf_meg = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

acoustic_splits = list(
    skf_acoustic.split(
        X_acoustic_raw,
        y_acoustic
    )
)

meg_splits = list(
    skf_meg.split(
        X_meg_raw,
        y_meg
    )
)


# ============================================================
# INITIALIZE OOF ARRAYS
# ============================================================

oof_pred_acoustic = np.zeros(
    len(y_acoustic),
    dtype=np.int64
)

oof_prob_acoustic = np.zeros(
    len(y_acoustic),
    dtype=np.float32
)

oof_pred_meg = np.zeros(
    len(y_meg),
    dtype=np.int64
)

oof_prob_meg = np.zeros(
    len(y_meg),
    dtype=np.float32
)

oof_count_acoustic = np.zeros(
    len(y_acoustic),
    dtype=np.int64
)

oof_count_meg = np.zeros(
    len(y_meg),
    dtype=np.int64
)

fused_acoustic_32D = np.zeros(
    (len(y_acoustic), LATENT_DIM),
    dtype=np.float32
)

fused_meg_32D = np.zeros(
    (len(y_meg), LATENT_DIM),
    dtype=np.float32
)

fold_records = []
selected_features_by_fold = {}


print("\n" + "=" * 72)
print(
    "A3: RESIDUAL ENCODER + DANN "
    "| Residual ON | DANN/GRL ON | SCL OFF"
)
print("=" * 72)


# ============================================================
# OUTER FOLD LOOP
# ============================================================

for fold in range(N_SPLITS):

    fold_no = fold + 1

    print("\n" + "=" * 72)
    print(f"A3 OUTER FOLD {fold_no}/{N_SPLITS}")
    print("=" * 72)

    # --------------------------------------------------------
    # 1. TRAIN/TEST INDICES
    # --------------------------------------------------------

    train_a_idx, test_a_idx = acoustic_splits[fold]
    train_m_idx, test_m_idx = meg_splits[fold]

    Xa_tr_raw = X_acoustic_raw[train_a_idx]
    Xa_te_raw = X_acoustic_raw[test_a_idx]

    ya_tr = y_acoustic[train_a_idx]
    ya_te = y_acoustic[test_a_idx]

    Xm_tr_raw = X_meg_raw[train_m_idx]
    Xm_te_raw = X_meg_raw[test_m_idx]

    ym_tr = y_meg[train_m_idx]
    ym_te = y_meg[test_m_idx]

    print(
        f"Acoustic train/test: "
        f"{len(train_a_idx)}/{len(test_a_idx)}"
    )

    print(
        f"MEG train/test     : "
        f"{len(train_m_idx)}/{len(test_m_idx)}"
    )

    # --------------------------------------------------------
    # 2. TRAIN-FOLD-ONLY PREPROCESSING
    # --------------------------------------------------------

    (
        Xa_tr,
        Xa_te,
        acoustic_medians,
        acoustic_lower,
        acoustic_upper
    ) = preprocess_fold_train_test(
        Xa_tr_raw,
        Xa_te_raw
    )

    (
        Xm_tr,
        Xm_te,
        meg_medians,
        meg_lower,
        meg_upper
    ) = preprocess_fold_train_test(
        Xm_tr_raw,
        Xm_te_raw
    )

    # --------------------------------------------------------
    # 3. RF-SHAP TOP-7: ACOUSTIC
    # --------------------------------------------------------

    a_idx, a_names = select_top_features_rf_shap(
        X_train=Xa_tr,
        y_train=ya_tr,
        feature_names=acoustic_feature_names,
        top_k=TOP_K,
        random_state=SEED + fold_no,
        dataset_name=f"Acoustic fold {fold_no}"
    )

    Xa_tr = Xa_tr[:, a_idx]
    Xa_te = Xa_te[:, a_idx]

    # --------------------------------------------------------
    # 4. RF-SHAP TOP-7: MEG
    # --------------------------------------------------------

    m_idx, m_names = select_top_features_rf_shap(
        X_train=Xm_tr,
        y_train=ym_tr,
        feature_names=meg_feature_names,
        top_k=TOP_K,
        random_state=SEED + 100 + fold_no,
        dataset_name=f"MEG fold {fold_no}"
    )

    Xm_tr = Xm_tr[:, m_idx]
    Xm_te = Xm_te[:, m_idx]

    if len(a_idx) != len(m_idx):
        raise ValueError(
            "Acoustic and MEG must have equal selected "
            "feature counts for the shared encoder."
        )

    selected_features_by_fold[fold_no] = {
        "acoustic": a_names,
        "meg": m_names
    }

    # --------------------------------------------------------
    # 5. STANDARDIZATION
    # Fit scalers on training folds only.
    # --------------------------------------------------------

    scaler_acoustic = StandardScaler()

    Xa_tr = scaler_acoustic.fit_transform(
        Xa_tr
    ).astype(np.float32)

    Xa_te = scaler_acoustic.transform(
        Xa_te
    ).astype(np.float32)

    scaler_meg = StandardScaler()

    Xm_tr = scaler_meg.fit_transform(
        Xm_tr
    ).astype(np.float32)

    Xm_te = scaler_meg.transform(
        Xm_te
    ).astype(np.float32)

    for name, arr in [
        ("Acoustic train", Xa_tr),
        ("Acoustic test", Xa_te),
        ("MEG train", Xm_tr),
        ("MEG test", Xm_te)
    ]:

        if not np.isfinite(arr).all():
            raise RuntimeError(
                f"{name} contains non-finite values "
                f"in fold {fold_no}."
            )

    # --------------------------------------------------------
    # 6. TRAIN RESIDUAL ENCODER + DANN
    # SCL is not used.
    # --------------------------------------------------------

    model = train_dann(
        X_acoustic_train=Xa_tr,
        y_acoustic_train=ya_tr,
        X_meg_train=Xm_tr,
        y_meg_train=ym_tr,
        fold_number=fold_no
    )

    # --------------------------------------------------------
    # 7. EXTRACT 32-D LATENT REPRESENTATIONS
    # --------------------------------------------------------

    za_tr = extract_latent(
        model,
        Xa_tr
    )

    zm_tr = extract_latent(
        model,
        Xm_tr
    )

    za_te = extract_latent(
        model,
        Xa_te
    )

    zm_te = extract_latent(
        model,
        Xm_te
    )

    X_latent_train = np.vstack([
        za_tr,
        zm_tr
    ]).astype(np.float32)

    y_latent_train = np.hstack([
        ya_tr,
        ym_tr
    ]).astype(np.int64)

    X_latent_test = np.vstack([
        za_te,
        zm_te
    ]).astype(np.float32)

    y_latent_test = np.hstack([
        ya_te,
        ym_te
    ]).astype(np.int64)

    # --------------------------------------------------------
    # 8. TRAIN FINAL MLP AND EVALUATE HELD-OUT PARTICIPANTS
    # --------------------------------------------------------

    result = train_and_evaluate_mlp(
        X_train=X_latent_train,
        y_train=y_latent_train,
        X_test=X_latent_test,
        y_test=y_latent_test,
        fold_number=fold_no
    )

    fold_records.append({
        "Fold": fold_no,
        **{
            key: result[key]
            for key in [
                "Accuracy",
                "Precision",
                "Recall",
                "Specificity",
                "F1",
                "AUC",
                "TN",
                "FP",
                "FN",
                "TP"
            ]
        }
    })

    # --------------------------------------------------------
    # 9. STORE HELD-OUT PREDICTIONS AND EMBEDDINGS
    # --------------------------------------------------------

    n_a_test = len(test_a_idx)

    pred_a_test = result["predictions"][:n_a_test]
    prob_a_test = result["probabilities"][:n_a_test]

    pred_m_test = result["predictions"][n_a_test:]
    prob_m_test = result["probabilities"][n_a_test:]

    oof_pred_acoustic[test_a_idx] = pred_a_test
    oof_prob_acoustic[test_a_idx] = prob_a_test

    oof_pred_meg[test_m_idx] = pred_m_test
    oof_prob_meg[test_m_idx] = prob_m_test

    fused_acoustic_32D[test_a_idx] = za_te
    fused_meg_32D[test_m_idx] = zm_te

    oof_count_acoustic[test_a_idx] += 1
    oof_count_meg[test_m_idx] += 1

    print("\nHeld-out latent shapes")
    print("Acoustic:", za_te.shape)
    print("MEG     :", zm_te.shape)


# ============================================================
# OOF ASSIGNMENT CHECK
# ============================================================

print("\n" + "=" * 72)
print("OOF ASSIGNMENT CHECK")
print("=" * 72)

print(
    "Acoustic OOF counts:",
    np.unique(
        oof_count_acoustic,
        return_counts=True
    )
)

print(
    "MEG OOF counts:",
    np.unique(
        oof_count_meg,
        return_counts=True
    )
)

assert np.all(oof_count_acoustic == 1)
assert np.all(oof_count_meg == 1)


# ============================================================
# COMBINE OOF EMBEDDINGS AND PREDICTIONS
# ============================================================

X_fused = np.vstack([
    fused_acoustic_32D,
    fused_meg_32D
]).astype(np.float32)

y_fused = np.hstack([
    y_acoustic,
    y_meg
]).astype(np.int64)

fused_oof_probs = np.hstack([
    oof_prob_acoustic,
    oof_prob_meg
]).astype(np.float32)

fused_oof_preds = np.hstack([
    oof_pred_acoustic,
    oof_pred_meg
]).astype(np.int64)


# ============================================================
# COMBINED OOF METRICS
# ============================================================

print("\n" + "=" * 72)
print("A3 COMBINED OOF RESULTS")
print("=" * 72)

print(
    f"Accuracy  : "
    f"{accuracy_score(y_fused, fused_oof_preds):.4f}"
)

print(
    f"Precision : "
    f"{precision_score(y_fused, fused_oof_preds, zero_division=0):.4f}"
)

print(
    f"Recall    : "
    f"{recall_score(y_fused, fused_oof_preds, zero_division=0):.4f}"
)

print(
    f"F1-score  : "
    f"{f1_score(y_fused, fused_oof_preds, zero_division=0):.4f}"
)

print(
    f"AUC       : "
    f"{roc_auc_score(y_fused, fused_oof_probs):.4f}"
)


# ============================================================
# MEAN AND STANDARD DEVIATION ACROSS FOLDS
# ============================================================

fold_metrics_df = pd.DataFrame(
    fold_records
)

print("\nA3 per-fold metrics:")
print(
    fold_metrics_df.round(4).to_string(index=False)
)

summary_rows = []

for metric in [
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1",
    "AUC"
]:

    values = fold_metrics_df[
        metric
    ].to_numpy(dtype=float)

    summary_rows.append({
        "Metric": metric,
        "Mean": np.nanmean(values),
        "SD": np.nanstd(values, ddof=1)
    })

summary_df = pd.DataFrame(
    summary_rows
)

print("\nA3 mean and SD across folds:")
print(
    summary_df.round(4).to_string(index=False)
)


# ============================================================
# FINAL OOF LATENT SHAPES
# ============================================================

print("\n" + "=" * 72)
print("A3 FINAL OOF LATENT SHAPES")
print("=" * 72)

print(
    "Acoustic 32-D:",
    fused_acoustic_32D.shape
)

print(
    "MEG 32-D     :",
    fused_meg_32D.shape
)

print(
    "Fused 32-D   :",
    X_fused.shape
)

assert fused_acoustic_32D.shape == (
    len(y_acoustic),
    LATENT_DIM
)

assert fused_meg_32D.shape == (
    len(y_meg),
    LATENT_DIM
)

assert X_fused.shape == (
    len(y_acoustic) + len(y_meg),
    LATENT_DIM
)

assert np.isfinite(X_fused).all()


# ============================================================
# SAVE OOF EMBEDDINGS AND PREDICTIONS
# ============================================================

latent_columns = [
    f"latent_{i+1}"
    for i in range(LATENT_DIM)
]

oof_acoustic_df = pd.DataFrame(
    fused_acoustic_32D,
    columns=latent_columns
)

oof_acoustic_df["Status"] = y_acoustic
oof_acoustic_df["OOF_Prediction"] = oof_pred_acoustic
oof_acoustic_df["OOF_Probability"] = oof_prob_acoustic

oof_meg_df = pd.DataFrame(
    fused_meg_32D,
    columns=latent_columns
)

oof_meg_df["Status"] = y_meg
oof_meg_df["OOF_Prediction"] = oof_pred_meg
oof_meg_df["OOF_Probability"] = oof_prob_meg

oof_acoustic_df.to_csv(
    "mira_net_A3_acoustic_oof_embeddings.csv",
    index=False
)

oof_meg_df.to_csv(
    "mira_net_A3_meg_oof_embeddings.csv",
    index=False
)

fold_metrics_df.to_csv(
    "mira_net_A3_fold_metrics.csv",
    index=False
)

summary_df.to_csv(
    "mira_net_A3_fold_summary.csv",
    index=False
)


# ============================================================
# SAVE FOLD-SPECIFIC RF-SHAP FEATURES
# ============================================================

feature_rows = []

for fold_no, modalities in selected_features_by_fold.items():

    for modality in ["acoustic", "meg"]:

        for rank, feature_name in enumerate(
            modalities[modality],
            start=1
        ):

            feature_rows.append({
                "Fold": fold_no,
                "Modality": modality,
                "Rank": rank,
                "Feature": feature_name
            })

pd.DataFrame(
    feature_rows
).to_csv(
    "mira_net_A3_rf_shap_selected_features.csv",
    index=False
)


# ============================================================
# COMPLETION SUMMARY
# ============================================================

print("\n" + "=" * 72)
print("A3 ABLATION COMPLETE")
print("=" * 72)

print("Residual block: ON")
print("DANN / GRL    : ON")
print("SCL           : OFF")

print(
    "Encoder epochs:",
    ENCODER_EPOCHS,
    "| Final MLP epochs:",
    MLP_EPOCHS
)

print(
    "Acoustic 32-D:",
    fused_acoustic_32D.shape
)

print(
    "MEG 32-D     :",
    fused_meg_32D.shape
)

print(
    "Combined 32-D:",
    X_fused.shape
)

print("\nOutput files:")
print(" - mira_net_A3_acoustic_oof_embeddings.csv")
print(" - mira_net_A3_meg_oof_embeddings.csv")
print(" - mira_net_A3_fold_metrics.csv")
print(" - mira_net_A3_fold_summary.csv")
print(" - mira_net_A3_rf_shap_selected_features.csv")

Device: cpu

Dataset shapes
Acoustic: (193, 69)
MEG     : (127, 105)
Acoustic labels: {np.int64(0): np.int64(66), np.int64(1): np.int64(127)}
MEG labels: {np.int64(0): np.int64(65), np.int64(1): np.int64(62)}

A3: RESIDUAL ENCODER + DANN | Residual ON | DANN/GRL ON | SCL OFF

A3 OUTER FOLD 1/5
Acoustic train/test: 154/39
MEG train/test     : 101/26

Acoustic fold 1 RF-SHAP Top-7 features:
01. Degree-voice breaks (mean |SHAP|=0.020099)
02. MDVP:Shimmer (mean |SHAP|=0.019161)
03. Delta1 (mean |SHAP|=0.019100)
04. RPDE (mean |SHAP|=0.018424)
05. Delta0 (mean |SHAP|=0.017792)
06. No. of voice breaks (mean |SHAP|=0.017317)
07. MFCC5 (mean |SHAP|=0.017233)

MEG fold 1 RF-SHAP Top-7 features:
01. burst_rate_asym (mean |SHAP|=0.019693)
02. burst_dur_cv_R_std (mean |SHAP|=0.014527)
03. burst_dur_cv_L_std (mean |SHAP|=0.013783)
04. ibi_cv_asym (mean |SHAP|=0.013600)
05. mean_burst_duration_R_std (mean |SHAP|=0.013082)
06. mean_ibi_R_std (mean |SHAP|=0.012419)
07. mean_burst_duration_asym (mean |

In [5]:
import torch

# Assuming fused_acoustic_32D and fused_meg_32D are numpy arrays of shape (7350, 32)

# Step 1: Convert numpy arrays to torch tensors (float32)
fused_acoustic_tensor = torch.from_numpy(fused_acoustic_32D).float()
fused_meg_tensor = torch.from_numpy(fused_meg_32D).float()

# Step 2: Concatenate row-wise (along dim=0)
fused_output_tensor = torch.cat([fused_acoustic_tensor, fused_meg_tensor], dim=0)  # Shape: (14700, 32)

# Step 3: Print shape and dtype
print(fused_output_tensor.shape)  # torch.Size([14700, 32])
print(fused_output_tensor.dtype)  # torch.float32


torch.Size([320, 32])
torch.float32


In [6]:
import torch
import numpy as np

# ------------------------
# Step 1: Extract targets correctly
# ------------------------

# Acoustic target from df_acoustic_top7 (last column)
y_acoustic = acoustic_dataset.iloc[:, -1].values.astype(np.int32)   # (7350,)

# MEG target from final_meg_df (last column)
y_meg = meg_dataset.iloc[:, -1].values.astype(np.int32)           # (7350,)

# Convert to tensors (reshape to column)
acoustic_target_tensor = torch.tensor(y_acoustic).long().reshape(-1, 1)   # (7350, 1)
meg_target_tensor      = torch.tensor(y_meg).long().reshape(-1, 1)        # (7350, 1)

# ------------------------
# Step 2: Convert 32D latent features to tensors
# ------------------------

fused_acoustic_tensor = torch.from_numpy(fused_acoustic_32D).float()  # (7350, 32)
fused_meg_tensor      = torch.from_numpy(fused_meg_32D).float()       # (7350, 32)

# ------------------------
# Step 3: Attach target columns
# ------------------------

# Concatenate latent features with target
acoustic_full = torch.cat([fused_acoustic_tensor, acoustic_target_tensor.float()], dim=1)  # (7350, 65)
meg_full      = torch.cat([fused_meg_tensor,      meg_target_tensor.float()],      dim=1)  # (7350, 65)

# ------------------------
# Step 4: Concatenate both datasets
# ------------------------

fused_df = torch.cat([acoustic_full, meg_full], dim=0)  # (14700, 65)

# ------------------------
# Step 5: Separate features and labels for PyTorch
# ------------------------

X = fused_df[:, :-1].float()   # Features (32D) -> float32
y = fused_df[:, -1].long()     # Target -> int32

# ------------------------
# Step 6: Print shapes and dtypes
# ------------------------

print(f"Features shape: {X.shape}, dtype: {X.dtype}")  # (14700, 32), float32
print(f"Labels shape: {y.shape}, dtype: {y.dtype}")    # (14700,), int32

Features shape: torch.Size([320, 32]), dtype: torch.float32
Labels shape: torch.Size([320]), dtype: torch.int64


In [7]:
import torch
import pandas as pd

# fused_df shape = (14700, 33)

# Step 6: Convert fused tensor to numpy
fused_df_np = fused_df.numpy()

# Step 7: Create correct column names for 32 latent features + 1 label
feature_names = [f"feat_{i}" for i in range(32)]
feature_names.append("PD_Label_Combined")

# Step 8: Convert to DataFrame
fused_df_named = pd.DataFrame(fused_df_np, columns=feature_names)

# Step 9: Verify
print(fused_df_named.shape)
print(fused_df_named.head())
print(fused_df_named.tail())

(320, 33)
     feat_0    feat_1    feat_2    feat_3    feat_4    feat_5    feat_6  \
0  0.071220 -0.220263  0.122276  0.237670 -0.160891  0.187094  0.056466   
1 -0.078713 -0.022329  0.326928  0.086238 -0.146731  0.168750  0.033203   
2 -0.274826  0.051433  0.237766 -0.291377 -0.142246  0.060824 -0.076594   
3  0.087228 -0.286881 -0.047168 -0.013220 -0.194479 -0.150760  0.058670   
4  0.038244  0.224110  0.121077 -0.272419  0.236390 -0.015518 -0.205193   

     feat_7    feat_8    feat_9  ...   feat_23   feat_24   feat_25   feat_26  \
0  0.141062 -0.021010 -0.140726  ...  0.094946 -0.209317 -0.224735  0.083965   
1 -0.009072 -0.165335  0.027765  ...  0.220999 -0.235034 -0.044325  0.030218   
2 -0.079431 -0.101639  0.037834  ... -0.180754  0.184182 -0.350408  0.102821   
3 -0.203154 -0.097000  0.143574  ...  0.250735 -0.023610 -0.059297  0.292139   
4  0.124688  0.205992  0.142444  ...  0.134807  0.220105 -0.163901  0.160312   

    feat_27   feat_28   feat_29   feat_30   feat_31  PD_La

In [8]:
# Make a copy to be safe (optional)
fused_df_named_int = fused_df_named.copy()

# Convert only the last column to int
fused_df_named_int['PD_Label_Combined'] = fused_df_named_int['PD_Label_Combined'].astype(int)

# Verify
print(fused_df_named_int.dtypes)
print(fused_df_named_int.head())

feat_0               float32
feat_1               float32
feat_2               float32
feat_3               float32
feat_4               float32
feat_5               float32
feat_6               float32
feat_7               float32
feat_8               float32
feat_9               float32
feat_10              float32
feat_11              float32
feat_12              float32
feat_13              float32
feat_14              float32
feat_15              float32
feat_16              float32
feat_17              float32
feat_18              float32
feat_19              float32
feat_20              float32
feat_21              float32
feat_22              float32
feat_23              float32
feat_24              float32
feat_25              float32
feat_26              float32
feat_27              float32
feat_28              float32
feat_29              float32
feat_30              float32
feat_31              float32
PD_Label_Combined      int64
dtype: object
     feat_0    feat_1    feat

In [9]:

# ============================================================
# IMPROVED ENSEMBLE CLASSIFICATION WITH 95% CONFIDENCE INTERVALS
# Stratified 10-Fold CV + Independent Test Set
#
# Models:
#   1. XGBoost
#   2. Random Forest
#   3. HistGradientBoosting
#   4. Weighted Soft Voting
#   5. Stacking
#
# Metrics with 95% CI:
# Accuracy, Precision, Sensitivity, Specificity, F1, AUC
# ============================================================

!pip install -q xgboost scikit-learn statsmodels pandas numpy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.impute import SimpleImputer
from sklearn.base import clone

from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix
)

from statsmodels.stats.proportion import proportion_confint
from xgboost import XGBClassifier


# ============================================================
# STEP 1: PREPARE DATA
# ============================================================

feature_cols = [f"feat_{i}" for i in range(32)]

X = fused_df_named_int[feature_cols].copy()
y = fused_df_named_int["PD_Label_Combined"].copy()

X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
y = pd.to_numeric(y, errors="coerce")

valid = y.notna()

X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].astype(int).reset_index(drop=True)

if set(y.unique()) != {0, 1}:
    raise ValueError("Labels must be encoded as 0 and 1.")

if X.shape[1] != 32:
    raise ValueError("Expected exactly 32 embedding features.")

if y.value_counts().min() < 10:
    raise ValueError(
        "Each class needs at least 10 samples for stratified "
        "10-fold cross-validation."
    )

print("X shape:", X.shape)
print("\nClass distribution:")
print(y.value_counts().sort_index())


# ============================================================
# STEP 2: TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    stratify=y,
    random_state=42
)

print("\nTraining shape:", X_train.shape)
print("Test shape:", X_test.shape)


# ============================================================
# STEP 3: MODEL FACTORY
# ============================================================

def create_models(y_fit):

    n_negative = int((y_fit == 0).sum())
    n_positive = int((y_fit == 1).sum())

    scale_pos_weight = n_negative / max(n_positive, 1)

    xgb = XGBClassifier(
        n_estimators=2500,
        max_depth=3,
        learning_rate=0.03,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.85,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=5.0,
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    rf = RandomForestClassifier(
        n_estimators=2500,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    hgb = HistGradientBoostingClassifier(
        max_iter=2500,
        learning_rate=0.04,
        max_leaf_nodes=15,
        max_depth=5,
        min_samples_leaf=10,
        l2_regularization=2.0,
        class_weight="balanced",
        early_stopping=False,
        random_state=42
    )

    voting = VotingClassifier(
        estimators=[
            ("xgb", clone(xgb)),
            ("rf", clone(rf)),
            ("hgb", clone(hgb))
        ],
        voting="soft",
        weights=[2, 1, 1],
        n_jobs=1
    )

    stacking = StackingClassifier(
        estimators=[
            ("xgb", clone(xgb)),
            ("rf", clone(rf)),
            ("hgb", clone(hgb))
        ],
        final_estimator=LogisticRegression(
            C=0.5,
            max_iter=2000,
            class_weight="balanced"
        ),
        stack_method="predict_proba",
        passthrough=False,
        cv=StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=123
        ),
        n_jobs=1
    )

    return {
        "XGBoost": xgb,
        "Random Forest": rf,
        "HistGradientBoosting": hgb,
        "Voting Fusion": voting,
        "Stacking Fusion": stacking
    }


# ============================================================
# STEP 4: METRICS + 95% CONFIDENCE INTERVALS
# ============================================================

def calculate_metrics_with_ci(
    y_true,
    y_pred,
    y_prob,
    n_boot=2000,
    random_state=42
):

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    n = len(y_true)

    # ---------------- Point estimates ----------------

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true, y_pred, zero_division=0
    )

    sensitivity = recall_score(
        y_true, y_pred, zero_division=0
    )

    specificity = (
        tn / (tn + fp) if (tn + fp) > 0 else np.nan
    )

    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )

    auc = roc_auc_score(y_true, y_prob)

    results = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "F1": f1,
        "AUC": auc,
        "Balanced Accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

    # ---------------- Wilson 95% CIs ----------------

    proportions = {
        "Accuracy": (tp + tn, n),
        "Precision": (tp, tp + fp),
        "Sensitivity": (tp, tp + fn),
        "Specificity": (tn, tn + fp)
    }

    for metric, (count, denominator) in proportions.items():

        if denominator > 0:
            low, high = proportion_confint(
                count=count,
                nobs=denominator,
                alpha=0.05,
                method="wilson"
            )
        else:
            low, high = np.nan, np.nan

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    # ---------------- Bootstrap CIs for F1 and AUC ----------------
    # Stratified resampling preserves both class counts.
    # Predictions and probabilities remain paired with labels.

    rng = np.random.default_rng(random_state)

    class_indices = [
        np.flatnonzero(y_true == 0),
        np.flatnonzero(y_true == 1)
    ]

    bootstrap_f1 = []
    bootstrap_auc = []

    for _ in range(n_boot):

        sampled_indices = np.concatenate([
            rng.choice(
                indices,
                size=len(indices),
                replace=True
            )
            for indices in class_indices
        ])

        yt = y_true[sampled_indices]
        yp = y_pred[sampled_indices]
        pr = y_prob[sampled_indices]

        bootstrap_f1.append(
            f1_score(yt, yp, zero_division=0)
        )

        bootstrap_auc.append(
            roc_auc_score(yt, pr)
        )

    for metric, values in [
        ("F1", bootstrap_f1),
        ("AUC", bootstrap_auc)
    ]:

        low, high = np.percentile(
            values,
            [2.5, 97.5]
        )

        results[f"{metric} CI Low"] = low
        results[f"{metric} CI High"] = high

    return results


# ============================================================
# STEP 5: PREPROCESSING
# ============================================================

def make_preprocessor():
    return SimpleImputer(strategy="median")


# ============================================================
# STEP 6: STRATIFIED 10-FOLD CROSS-VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

model_names = [
    "XGBoost",
    "Random Forest",
    "HistGradientBoosting",
    "Voting Fusion",
    "Stacking Fusion"
]

cv_records = []
cv_summary_records = []
test_records = []
confusion_matrices = {}

metrics_to_report = [
    "Accuracy",
    "Precision",
    "Sensitivity",
    "Specificity",
    "F1",
    "AUC"
]


for model_name in model_names:

    print("\n" + "=" * 85)
    print("EVALUATING:", model_name)
    print("=" * 85)

    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1
    ):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        # Fit imputation only on this fold's training partition.
        imputer = make_preprocessor()

        X_tr_imp = imputer.fit_transform(X_tr)
        X_val_imp = imputer.transform(X_val)

        models = create_models(y_tr)
        model = models[model_name]

        model.fit(X_tr_imp, y_tr)

        val_pred = model.predict(X_val_imp)
        val_prob = model.predict_proba(X_val_imp)[:, 1]

        metrics = calculate_metrics_with_ci(
            y_val,
            val_pred,
            val_prob,
            n_boot=2000,
            random_state=1000 + fold
        )

        metrics["Model"] = model_name
        metrics["Fold"] = fold

        fold_metrics.append(metrics)
        cv_records.append(metrics.copy())

        # Print all six metrics and their CIs for each fold.
        print(f"\nFold {fold:02d}/10")

        for metric in metrics_to_report:

            print(
                f"{metric:15s}: {metrics[metric]:.4f} "
                f"(95% CI: "
                f"{metrics[metric + ' CI Low']:.4f}-"
                f"{metrics[metric + ' CI High']:.4f})"
            )

    # ========================================================
    # STEP 7: CV MEAN AND STANDARD DEVIATION
    # ========================================================

    fold_df = pd.DataFrame(fold_metrics)

    print(f"\n{model_name} - 10-Fold CV Summary")
    print("-" * 75)

    summary = {"Model": model_name}

    for metric in metrics_to_report:

        mean_val = fold_df[metric].mean()
        std_val = fold_df[metric].std(ddof=1)

        # Average lower and upper fold CI endpoints.
        mean_ci_low = fold_df[
            f"{metric} CI Low"
        ].mean()

        mean_ci_high = fold_df[
            f"{metric} CI High"
        ].mean()

        summary[f"CV {metric} Mean"] = mean_val
        summary[f"CV {metric} SD"] = std_val
        summary[f"Mean Fold {metric} CI Low"] = mean_ci_low
        summary[f"Mean Fold {metric} CI High"] = mean_ci_high

        print(
            f"{metric:15s}: {mean_val:.4f} +/- {std_val:.4f} "
            f"| Mean fold CI: "
            f"{mean_ci_low:.4f}-{mean_ci_high:.4f}"
        )

    for metric in ["Balanced Accuracy", "MCC"]:

        summary[f"CV {metric} Mean"] = fold_df[metric].mean()
        summary[f"CV {metric} SD"] = fold_df[metric].std(ddof=1)

    cv_summary_records.append(summary)

    # ========================================================
    # STEP 8: FINAL MODEL AND INDEPENDENT TEST SET
    # ========================================================

    final_imputer = make_preprocessor()

    X_train_imp = final_imputer.fit_transform(X_train)
    X_test_imp = final_imputer.transform(X_test)

    final_models = create_models(y_train)
    final_model = final_models[model_name]

    final_model.fit(X_train_imp, y_train)

    test_pred = final_model.predict(X_test_imp)
    test_prob = final_model.predict_proba(X_test_imp)[:, 1]

    test_metrics = calculate_metrics_with_ci(
        y_test,
        test_pred,
        test_prob,
        n_boot=5000,
        random_state=2026
    )

    test_metrics["Model"] = model_name
    test_records.append(test_metrics.copy())

    confusion_matrices[model_name] = confusion_matrix(
        y_test,
        test_pred,
        labels=[0, 1]
    )

    print(f"\n{model_name} - Independent Test Set")
    print("-" * 75)

    for metric in metrics_to_report:

        print(
            f"{metric:15s}: {test_metrics[metric]:.4f} "
            f"(95% CI: "
            f"{test_metrics[metric + ' CI Low']:.4f}-"
            f"{test_metrics[metric + ' CI High']:.4f})"
        )

    print(
        f"Balanced Accuracy: "
        f"{test_metrics['Balanced Accuracy']:.4f}"
    )

    print(f"MCC: {test_metrics['MCC']:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrices[model_name])


# ============================================================
# STEP 9: INDIVIDUAL FOLD RESULTS WITH 95% CIs
# ============================================================

fold_results_df = pd.DataFrame(cv_records)

fold_columns = [
    "Model", "Fold",
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

fold_results_df = fold_results_df[fold_columns]

print("\n" + "=" * 100)
print("ALL MODELS: INDIVIDUAL FOLD RESULTS WITH 95% CIs")
print("=" * 100)

display(
    fold_results_df.sort_values(
        ["Model", "Fold"]
    ).round(4)
)


# ============================================================
# STEP 10: CROSS-VALIDATION SUMMARY
# ============================================================

cv_summary_df = pd.DataFrame(cv_summary_records)

print("\n" + "=" * 100)
print("CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs")
print("=" * 100)

display(cv_summary_df.round(4))


# ============================================================
# STEP 11: INDEPENDENT TEST RESULTS WITH 95% CIs
# ============================================================

test_results_df = pd.DataFrame(
    test_records
).set_index("Model")

test_columns = [
    "Accuracy", "Accuracy CI Low", "Accuracy CI High",
    "Precision", "Precision CI Low", "Precision CI High",
    "Sensitivity", "Sensitivity CI Low", "Sensitivity CI High",
    "Specificity", "Specificity CI Low", "Specificity CI High",
    "F1", "F1 CI Low", "F1 CI High",
    "AUC", "AUC CI Low", "AUC CI High",
    "Balanced Accuracy", "MCC",
    "TN", "FP", "FN", "TP"
]

test_results_df = test_results_df[test_columns]

print("\n" + "=" * 100)
print("INDEPENDENT TEST SET COMPARISON WITH 95% CIs")
print("=" * 100)

display(
    test_results_df.sort_values(
        by="F1",
        ascending=False
    ).round(4)
)


# ============================================================
# STEP 12: VOTING ENSEMBLE RESULTS
# ============================================================

print("\n" + "=" * 100)
print("WEIGHTED VOTING ENSEMBLE RESULTS")
print("=" * 100)

print("Voting weights: XGBoost=2, Random Forest=1, HGB=1")

print("\nVoting fold results:")
display(
    fold_results_df[
        fold_results_df["Model"] == "Voting Fusion"
    ].round(4)
)

print("\nVoting cross-validation summary:")
display(
    cv_summary_df[
        cv_summary_df["Model"] == "Voting Fusion"
    ].round(4)
)

print("\nVoting independent test results:")
display(
    test_results_df.loc[["Voting Fusion"]].round(4)
)

X shape: (320, 32)

Class distribution:
PD_Label_Combined
0    131
1    189
Name: count, dtype: int64

Training shape: (288, 32)
Test shape: (32, 32)

EVALUATING: XGBoost

Fold 01/10
Accuracy       : 0.6207 (95% CI: 0.4400-0.7731)
Precision      : 0.6500 (95% CI: 0.4329-0.8188)
Sensitivity    : 0.7647 (95% CI: 0.5274-0.9044)
Specificity    : 0.4167 (95% CI: 0.1933-0.6805)
F1             : 0.7027 (95% CI: 0.5556-0.8333)
AUC            : 0.6520 (95% CI: 0.4461-0.8580)

Fold 02/10
Accuracy       : 0.6552 (95% CI: 0.4735-0.8006)
Precision      : 0.7692 (95% CI: 0.4974-0.9182)
Sensitivity    : 0.5882 (95% CI: 0.3601-0.7839)
Specificity    : 0.7500 (95% CI: 0.4677-0.9111)
F1             : 0.6667 (95% CI: 0.4615-0.8390)
AUC            : 0.6765 (95% CI: 0.4608-0.8578)

Fold 03/10
Accuracy       : 0.6552 (95% CI: 0.4735-0.8006)
Precision      : 0.7333 (95% CI: 0.4805-0.8910)
Sensitivity    : 0.6471 (95% CI: 0.4130-0.8269)
Specificity    : 0.6667 (95% CI: 0.3906-0.8619)
F1             : 0.6875 (

,Model,Fold,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
20,HistGradientBoosting,1,0.7241,0.5428,0.8530,0.7368,0.5121,0.8819,0.8235,0.5897,...,0.8947,0.7353,0.5342,0.9069,0.7034,0.4216,7,5,3,14
21,HistGradientBoosting,2,0.5862,0.4074,0.7449,0.7273,0.4344,0.9025,0.4706,0.2617,...,0.7742,0.6422,0.4167,0.8431,0.6103,0.2239,9,3,9,8
22,HistGradientBoosting,3,0.6207,0.4400,0.7731,0.7143,0.4535,0.8828,0.5882,0.3601,...,0.8125,0.7598,0.5587,0.9167,0.6275,0.2512,8,4,7,10
23,HistGradientBoosting,4,0.7586,0.5789,0.8778,0.7778,0.5479,0.9100,0.8235,0.5897,...,0.9143,0.8333,0.6667,0.9608,0.7451,0.4976,8,4,3,14
24,HistGradientBoosting,5,0.6552,0.4735,0.8006,0.6667,0.4537,0.8281,0.8235,0.5897,...,0.8500,0.7206,0.5147,0.9020,0.6201,0.2647,5,7,3,14
25,HistGradientBoosting,6,0.6207,0.4400,0.7731,0.6667,0.4375,0.8372,0.7059,0.4687,...,0.8235,0.6667,0.4608,0.8580,0.6029,0.2090,6,6,5,12
26,HistGradientBoosting,7,0.5517,0.3755,0.7159,0.6111,0.3862,0.7969,0.6471,0.4130,...,0.7778,0.5049,0.2941,0.7304,0.5319,0.0647,5,7,6,11
27,HistGradientBoosting,8,0.6897,0.5077,0.8272,0.7222,0.4913,0.8750,0.7647,0.5274,...,0.8649,0.6961,0.4902,0.8725,0.6740,0.3533,7,5,4,13
28,HistGradientBoosting,9,0.6786,0.4934,0.8207,0.7500,0.5050,0.8982,0.7059,0.4687,...,0.8750,0.7540,0.5615,0.9146,0.6711,0.3378,7,4,5,12
29,HistGradientBoosting,10,0.8571,0.6851,0.9430,0.8421,0.6243,0.9448,0.9412,0.7302,...,0.9714,0.8289,0.6578,0.9733,0.8342,0.6990,8,3,1,16



CROSS-VALIDATION MEAN, SD AND MEAN FOLD CIs


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
0,XGBoost,0.6394,0.0585,0.4585,0.7874,0.7047,0.0818,0.4667,0.8625,0.6941,...,0.5214,0.8379,0.7000,0.0793,0.4972,0.8789,0.6285,0.0722,0.2588,0.1382
1,Random Forest,0.6703,0.0665,0.4891,0.8114,0.7158,0.0601,0.4821,0.8701,0.7412,...,0.5608,0.8570,0.7331,0.0765,0.5369,0.8982,0.6551,0.0685,0.3180,0.1386
2,HistGradientBoosting,0.6743,0.0896,0.4944,0.8129,0.7215,0.0642,0.4846,0.8757,0.7294,...,0.5535,0.8558,0.7142,0.0961,0.5155,0.8878,0.6621,0.0847,0.3323,0.1764
3,Voting Fusion,0.6635,0.0613,0.4822,0.8063,0.7184,0.0573,0.4795,0.8745,0.7118,...,0.5386,0.8519,0.7231,0.0790,0.5256,0.8913,0.6536,0.0593,0.3124,0.1183
4,Stacking Fusion,0.6565,0.0532,0.4750,0.8011,0.7411,0.0619,0.4871,0.8933,0.6529,...,0.4981,0.8436,0.7224,0.0702,0.5247,0.8911,0.6579,0.0544,0.3157,0.1079



INDEPENDENT TEST SET COMPARISON WITH 95% CIs


...

,Model,Fold,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
30,Voting Fusion,1,0.6897,0.5077,0.8272,0.7000,0.4810,0.8545,0.8235,0.5897,...,0.8750,0.7549,0.5539,0.9119,0.6618,0.3444,6,6,3,14
31,Voting Fusion,2,0.5862,0.4074,0.7449,0.6923,0.4237,0.8732,0.5294,0.3096,...,0.7879,0.6569,0.4411,0.8480,0.5980,0.1942,8,4,8,9
32,Voting Fusion,3,0.6552,0.4735,0.8006,0.7333,0.4805,0.8910,0.6471,0.4130,...,0.8423,0.7500,0.5539,0.9069,0.6569,0.3092,8,4,6,11
33,Voting Fusion,4,0.7241,0.5428,0.8530,0.7647,0.5274,0.9044,0.7647,0.5274,...,0.8947,0.8088,0.6225,0.9510,0.7157,0.4314,8,4,4,13
34,Voting Fusion,5,0.6897,0.5077,0.8272,0.6818,0.4732,0.8364,0.8824,0.6566,...,0.8718,0.7108,0.5049,0.8824,0.6495,0.3442,5,7,2,15
35,Voting Fusion,6,0.6207,0.4400,0.7731,0.6667,0.4375,0.8372,0.7059,0.4687,...,0.8235,0.7206,0.5293,0.8873,0.6029,0.2090,6,6,5,12
36,Voting Fusion,7,0.5517,0.3755,0.7159,0.6250,0.3864,0.8152,0.5882,0.3601,...,0.7778,0.5539,0.3431,0.7697,0.5441,0.0874,6,6,7,10
37,Voting Fusion,8,0.6897,0.5077,0.8272,0.7222,0.4913,0.8750,0.7647,0.5274,...,0.8649,0.6765,0.4608,0.8627,0.6740,0.3533,7,5,4,13
38,Voting Fusion,9,0.6786,0.4934,0.8207,0.7857,0.5241,0.9243,0.6471,0.4130,...,0.8667,0.7914,0.6150,0.9412,0.6872,0.3656,8,3,6,11
39,Voting Fusion,10,0.7500,0.5664,0.8732,0.8125,0.5699,0.9341,0.7647,0.5274,...,0.9143,0.8075,0.6310,0.9519,0.7460,0.4855,8,3,4,13



Voting cross-validation summary:


,Model,CV Accuracy Mean,CV Accuracy SD,Mean Fold Accuracy CI Low,Mean Fold Accuracy CI High,CV Precision Mean,CV Precision SD,Mean Fold Precision CI Low,Mean Fold Precision CI High,CV Sensitivity Mean,...,Mean Fold F1 CI Low,Mean Fold F1 CI High,CV AUC Mean,CV AUC SD,Mean Fold AUC CI Low,Mean Fold AUC CI High,CV Balanced Accuracy Mean,CV Balanced Accuracy SD,CV MCC Mean,CV MCC SD
3,Voting Fusion,0.6635,0.0613,0.4822,0.8063,0.7184,0.0573,0.4795,0.8745,0.7118,...,0.5386,0.8519,0.7231,0.079,0.5256,0.8913,0.6536,0.0593,0.3124,0.1183



Voting independent test results:


,Accuracy,Accuracy CI Low,Accuracy CI High,Precision,Precision CI Low,Precision CI High,Sensitivity,Sensitivity CI Low,Sensitivity CI High,Specificity,...,F1 CI High,AUC,AUC CI Low,AUC CI High,Balanced Accuracy,MCC,TN,FP,FN,TP
Model,,,,,,,,,,,,,,,,,,,,,
Voting Fusion,0.5938,0.4226,0.7448,0.6364,0.4295,0.8027,0.7368,0.5121,0.8819,0.3846,...,0.8095,0.6518,0.4494,0.8421,0.5607,0.1287,5,8,5,14
